# 08v1 - visual confirmation: does A3's centre-anchor-only choice and 224px resolution explain the mcl_injury/lateral_meniscus_tear/synovitis gap?

**Not a training run.** This is a cheap, CPU-only diagnostic notebook,
built to check two hypotheses *before* spending Kaggle GPU time on a
retrain (user's explicit call, 2026-08-28) - see
[[project-rsna-phase-status]] for the full reasoning this notebook is
checking.

**Background:** A2 v1's pooled 4-fold CV (`notebooks/06v2_...`) found 5
findings clustered at 0.61-0.66 gold macro-AUC, clearly below the other
7 (0.79-0.88): `mcl_injury` (0.6145), `oa_lateral_compartment` (0.6325),
`synovitis` (0.6559), `lateral_meniscus_tear` (0.6584),
`medial_meniscus_tear` (0.6635). A 2026-08-28 investigation (local data
+ live Kaggle forum + 2 real public notebooks, `RESOURCES.md`) narrowed
this to two candidate root causes worth checking against our own real
cache pixels, not just other people's notebooks:

1. **Resolution.** `wguesdon/rsna-knee-dinov2-at-meniscus-resolution`
   (`RESOURCES.md`) argues a `d`-mm feature needs pixel pitch `<= d/2` to
   survive resizing (Nyquist). This notebook checks that arithmetic
   against **our own real `cache_meta.json`** (previous check used their
   150mm-FOV numbers, not ours).
2. **Centre-anchor-only slice selection.** A3's `select_group()` defaults
   to `group_index=1` (centre anchor, 3 of the 9 cached slices per slot)
   per the 2026-08-26 "follow the forum" decision.
   `dreaddevelopment/knee-mri-twelve-findings-from-a-single-model`
   (`RESOURCES.md`) reports that excluding outer slices "measurably cost
   accuracy" specifically on the **collateral ligaments and lateral
   meniscus** - `mcl_injury` is a collateral ligament. This notebook
   visually checks our own cached pixels: does group 1 (centre) actually
   show less of the relevant anatomy than groups 0/2 (outer), for real
   gold-positive studies?

**What this notebook does NOT do:** identify the actual tear/lesion in
any image (neither of us is a radiologist) or run any model. It only
checks whether the anatomy of interest (collateral ligament region,
meniscus region) is visually present/absent across the 3 anchor groups,
and grounds the resolution arithmetic in our real numbers instead of a
borrowed 150mm-FOV example.

**Needs the published cache Dataset attached to a Kaggle kernel**
(multi-GB `.npy` files) - run on Kaggle and report back the real output
(the two saved PNGs + what you see in them), same convention as
`00v2`/`03v2`/`04v2`.

## Setup

Same `CACHE_DIR`/`RAW_DIR` convention as `04v2_slot_cache_integration.ipynb`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

_KAGGLE_RAW = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
ON_KAGGLE = _KAGGLE_RAW.exists()
if not ON_KAGGLE:
    raise RuntimeError(
        "This notebook needs the published cache Dataset attached to a "
        "Kaggle kernel (multi-GB .npy files) - run on Kaggle, not locally."
    )

RAW_DIR = _KAGGLE_RAW
CACHE_DIR = Path("/kaggle/input/datasets/alherma7/cache-stevenleehans-rsna/cache")
if not (CACHE_DIR / "cache_meta.json").exists():
    raise RuntimeError(
        f"{CACHE_DIR} has no cache_meta.json - check the Dataset is "
        "attached to this kernel and CACHE_DIR still matches its mount path."
    )

with open(CACHE_DIR / "cache_meta.json") as f:
    meta = json.load(f)

SLOT_NAMES = meta["slots"]
CROP_MM = meta["crop_mm"]
IMG = meta["img"]
MM_PER_PX = CROP_MM / IMG
print("CACHE_DIR:", CACHE_DIR)
print("slots:", SLOT_NAMES)
print(f"crop_mm={CROP_MM}, img={IMG}, mm_per_px={MM_PER_PX:.4f}")

## Hypothesis 1 - resolution, against our own real numbers

Nyquist: a `d`-mm feature needs pixel pitch `<= d/2` to survive
resizing. Meniscal tears measure 1-3mm (RESOURCES.md, the sourced
clinical range); a collateral ligament itself is much wider (a few mm)
but a *tear within it* is comparably thin. Computed from our own
`cache_meta.json`, not an assumed/borrowed field of view.

In [ ]:
for d_mm in [1.0, 2.0, 3.0, 5.0]:
    needed_mm_per_px = d_mm / 2
    resolves = MM_PER_PX <= needed_mm_per_px
    print(f"{d_mm:>4.1f} mm feature: needs <= {needed_mm_per_px:.3f} mm/px, "
          f"our cache has {MM_PER_PX:.3f} mm/px -> resolves: {resolves}")

## Select gold studies of interest

`mcl_injury` (9 gold positives - all of them, few enough to show every
one) and a sample of `lateral_meniscus_tear` gold positives (23 total,
sampled to 6 for a readable figure, seed 42 matching this project's
other sampling cells).

In [ ]:
train_csv = pd.read_csv(RAW_DIR / "train.csv")
label_cols = [c for c in train_csv.columns if c not in ("StudyInstanceUID", "Report")]
gold_mask = train_csv[label_cols].notna().all(axis=1)
gold = train_csv.loc[gold_mask].reset_index(drop=True)
assert len(gold) == 58, len(gold)

mcl_pos_ids = gold.loc[gold["MCL"] == 1, "StudyInstanceUID"].tolist()
print(f"gold MCL-positive studies: {len(mcl_pos_ids)}")
assert len(mcl_pos_ids) == 9, len(mcl_pos_ids)

lat_men_pos_all = gold.loc[gold["Lateral Meniscus"] == 1, "StudyInstanceUID"].tolist()
print(f"gold Lateral Meniscus-positive studies: {len(lat_men_pos_all)}")

rng = np.random.default_rng(42)
lat_men_pos_ids = list(rng.choice(lat_men_pos_all, size=min(6, len(lat_men_pos_all)), replace=False))
print(f"sampled for the visual check: {len(lat_men_pos_ids)}")

## Locate these studies in the cache shards

Reads only the small `_studies.csv` files, same pattern as `04v2`'s
4-shard coverage cell.

In [ ]:
shard_names = [f"train.s{i:02d}of04" for i in range(4)]
id_to_loc = {}
for shard in shard_names:
    s = pd.read_csv(CACHE_DIR / f"{shard}_studies.csv")
    for row_idx, study_id in enumerate(s["StudyInstanceUID"]):
        id_to_loc[study_id] = (shard, row_idx)

missing_mcl = [sid for sid in mcl_pos_ids if sid not in id_to_loc]
missing_men = [sid for sid in lat_men_pos_ids if sid not in id_to_loc]
assert not missing_mcl, f"MCL-positive studies missing from cache: {missing_mcl}"
assert not missing_men, f"Lateral-meniscus-positive studies missing from cache: {missing_men}"
print("all selected gold studies located in the cache - OK")

In [ ]:
cache_by_shard = {shard: np.load(CACHE_DIR / f"{shard}_cache.npy", mmap_mode="r") for shard in shard_names}


def slot_slices(study_id, slot_name):
    """Return the (9, 224, 224) slice stack for one study's one slot."""
    shard, row_idx = id_to_loc[study_id]
    slot_idx = SLOT_NAMES.index(slot_name)
    return cache_by_shard[shard][row_idx, slot_idx]

## Hypothesis 2 - visual check, MCL (`COR_T1` slot)

Coronal T1 is the standard sequence for reading collateral ligaments.
Middle channel of each of the 3 anchor groups (indices 1/4/7 of the
9-slice stack) - group 0 = low anchor, group 1 = centre anchor
(A3's current default), group 2 = high anchor. A 3mm scale bar (the
sourced meniscal-tear upper bound) is drawn on the first image using
this cache's real `mm_per_px`, so the comparison is against a concrete
physical scale, not just "does it look different."

**What to look for:** if the collateral ligament (a thin band running
along the joint margin) is visibly present in groups 0/2 but cut off or
absent in group 1, that supports hypothesis 2 - `group_index=1` is
discarding the anatomy this finding needs. If all 3 groups look
similarly (in)complete, hypothesis 2 is not supported by this check and
the gap is more likely resolution (hypothesis 1) or something else.

In [ ]:
import matplotlib.pyplot as plt


def plot_group_comparison(study_ids, slot_name, out_path, scale_bar_mm=3.0):
    n = len(study_ids)
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    if n == 1:
        axes = axes[None, :]
    group_labels = ["group 0 (low)", "group 1 (centre)", "group 2 (high)"]
    mid_channel = [1, 4, 7]  # middle slice of each 3-slice anchor group

    for row, study_id in enumerate(study_ids):
        slices = slot_slices(study_id, slot_name)  # (9, 224, 224)
        for col, mi in enumerate(mid_channel):
            axes[row, col].imshow(slices[mi], cmap="gray")
            axes[row, col].axis("off")
            if row == 0:
                axes[row, col].set_title(group_labels[col], fontsize=9)
        axes[row, 0].text(-14, IMG // 2, study_id[-8:], fontsize=7, rotation=90,
                           va="center", ha="center")

    bar_px = scale_bar_mm / MM_PER_PX
    x0, y0 = 10, IMG - 15
    axes[0, 0].plot([x0, x0 + bar_px], [y0, y0], color="red", linewidth=2)
    axes[0, 0].text(x0, y0 - 6, f"{scale_bar_mm:.0f} mm", color="red", fontsize=7)

    plt.suptitle(f"{slot_name} - centre vs. outer anchor groups", fontsize=11)
    plt.tight_layout()
    plt.savefig(out_path, dpi=100)
    plt.show()
    print(f"saved {out_path}")


plot_group_comparison(mcl_pos_ids, "COR_T1", "mcl_group_comparison.png")

## Hypothesis 2 - visual check, lateral meniscus (`SAG_FLUID_FS` slot)

Fluid-sensitive fat-sat sagittal is the standard primary sequence for
reading meniscus tears (fluid highlights a tear line). Same group
0/1/2 comparison, same scale bar.

In [ ]:
plot_group_comparison(lat_men_pos_ids, "SAG_FLUID_FS", "lateral_meniscus_group_comparison.png")

## Real output (run 2026-08-28, on Kaggle, cache Dataset attached)

**Setup check:** `cache_meta.json` real values: `crop_mm=130.0`,
`img=224`, `mm_per_px=0.5804` - matches this notebook's assumptions
exactly - OK.

**Hypothesis 1 (resolution), our own real numbers:**
```
1.0 mm feature: needs <= 0.500 mm/px, our cache has 0.580 mm/px -> resolves: False
2.0 mm feature: needs <= 1.000 mm/px, our cache has 0.580 mm/px -> resolves: True
3.0 mm feature: needs <= 1.500 mm/px, our cache has 0.580 mm/px -> resolves: True
5.0 mm feature: needs <= 2.500 mm/px, our cache has 0.580 mm/px -> resolves: True
```
Meniscal tears span 1-3mm - only the smallest end of that range is below
our resolution. A real but **partial** contributor, not the full
explanation for a gap this size (0.66 vs. 0.83 cluster mean).

**Gold study selection:** 9/9 gold `mcl_injury`-positive, 23 gold
`lateral_meniscus_tear`-positive (6 sampled), all located in the cache -
OK.

**Hypothesis 2 (centre-anchor-only), visual read of both saved PNGs**
(not committed to the repo - real patient MRI data, same policy as
`sample_slot_cache_grid.png`, see `.gitignore`):

- **New, unplanned finding:** 2/9 (22%) gold `mcl_injury`-positive
  studies have a **completely blank `COR_T1` slot** (all-zero pixels in
  all 3 anchor groups) - real cache behaviour (a missing slot is stored
  as zeros, already documented in `load_slot_cache_shard`'s docstring),
  not a notebook bug. A genuine, separate limitation from `group_index`:
  for those 2 studies the model has zero `COR_T1` signal no matter which
  anchor group is selected. Not filtered out in this version - see
  `08v2`.
- **MCL, `COR_T1`, 7 real rows:** a repeated pattern - **group 1
  (centre)** more often shows a busier/brighter view consistent with the
  intercondylar-notch region (cruciate-ligament territory), while
  **groups 0/2 (outer)** more often show a cleaner peripheral-compartment
  view.
- **Lateral meniscus, `SAG_FLUID_FS`, 6/6 real rows:** same pattern -
  group 1 repeatedly busier/brighter centrally (consistent with
  intercondylar/cruciate signal bleeding into the frame), groups 0/2
  cleaner with the meniscus wedge more clearly delineated (clearest in
  rows `08452608`, `39890254`, `20253309`).
- **Read caveat, stated plainly:** this is a single-observer, non-
  radiologist qualitative read of two images - not a formal diagnostic
  read. It is coherent with the measured AUC split (central `acl_injury`
  0.854 high; peripheral `mcl_injury`/meniscus tears 0.61-0.66 low) and
  gives a concrete anatomical mechanism (A3's `RSNA_WINDOW=0.35,0.65` is
  a *narrow* sampling window, so even groups 0/2 are only mildly
  off-centre - none of the 3 groups necessarily reaches the true lateral/
  medial periphery, but centre-only is the worst-positioned of the
  three for peripheral structures).

**Conclusion: both hypotheses got real, non-trivial support - resolution
partially, centre-anchor-selection more strongly - but neither is
proven with certainty from this single pass.** Per the user's explicit
call (2026-08-28: confirm more before retraining), a follow-up
`08v2_meniscus_mcl_slot_group_check.ipynb` adds mask-presence filtering
(no more blank rows), `medial_meniscus_tear`, and an `acl_injury`
negative control (same slot, a *central* structure - if group 1 looks
cleaner for ACL than it did here for MCL/meniscus, that is much stronger,
non-anecdotal support for hypothesis 2).